# Creovanta Bengali Voice Engine — Colab
প্রথমে নিজের/অনুমতিপ্রাপ্ত কণ্ঠের ১–৩০ সেকেন্ডের পরিষ্কার অডিও দিয়ে বাংলা ভয়েস ক্লোনিং পরীক্ষা করুন। এই নোটবুক Google Colab-এর অস্থায়ী GPU ব্যবহার করে; এটি স্থায়ী সার্ভার নয়।

## 1. GPU চালু করুন
Colab menu: Runtime → Change runtime type → T4 GPU (যদি ফ্রি হিসেবে পাওয়া যায়) → Save. তারপর নিচের সেল চালান। GPU না পেলে পরে আবার চেষ্টা করুন; পেইড runtime নির্বাচন করবেন না।

In [ ]:
import torch, os
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: raise RuntimeError('GPU পাওয়া যায়নি। Runtime settings থেকে GPU নির্বাচন করে আবার চালান।')

## 2. প্রয়োজনীয় সফটওয়্যার ইনস্টল করুন
ইনস্টলেশন কয়েক মিনিট লাগতে পারে।

In [ ]:
!apt-get -qq update && apt-get -qq install -y sox libsox-dev ffmpeg
!git clone --depth 1 --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd /content/CosyVoice
!pip install -q -r requirements.txt
!pip install -q -U "huggingface_hub>=0.30.0,<1.0.0" hf-xet soundfile

## 3. বাংলা CosyVoice3 মডেল ডাউনলোড করুন
মডেলটি বড়; প্রথমবার ডাউনলোডে সময় লাগবে।

In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
snapshot_download('kawshikbuet17/bengali-cosyvoice3-tts', local_dir=MODEL_DIR)
print('Model ready:', MODEL_DIR)

## 4. মডেল লোড করুন
এটি GPU memory ও runtime-এর ওপর নির্ভর করে কিছু সময় নিতে পারে।

In [ ]:
import sys, os, torch, torchaudio
sys.path.append('/content/CosyVoice/third_party/Matcha-TTS')
from cosyvoice.cli.cosyvoice import AutoModel
cosyvoice = AutoModel(model_dir=MODEL_DIR)
print('CosyVoice loaded. Sample rate:', cosyvoice.sample_rate)

## 5. নিজের কণ্ঠের নমুনা আপলোড করুন
শুধু নিজের কণ্ঠ বা স্পষ্ট অনুমতি পাওয়া ব্যক্তির কণ্ঠ ব্যবহার করুন। ১–৩০ সেকেন্ড, পরিষ্কার ও একক বক্তার অডিও সবচেয়ে ভালো।

In [ ]:
from google.colab import files
uploaded = files.upload()  # আপনার voice sample নির্বাচন করুন
PROMPT_PATH = '/content/' + next(iter(uploaded.keys()))
print('Uploaded:', PROMPT_PATH)

## 6. বাংলা লেখা থেকে কণ্ঠ তৈরি করুন
নিচের TEXT-এ নিজের বাংলা বাক্য লিখুন।

In [ ]:
import torchaudio
TEXT = 'আসসালামু আলাইকুম। এটি ক্রিওভান্টা বাংলা ভয়েস ক্লোনিংয়ের একটি পরীক্ষা।'
SYSTEM_PREFIX = 'You are a helpful assistant.<|endofprompt|>'
PROMPT_SR = 16000
audio, sr = torchaudio.load(PROMPT_PATH)
if audio.shape[0] > 1: audio = audio.mean(dim=0, keepdim=True)
if sr != PROMPT_SR: audio = torchaudio.functional.resample(audio, sr, PROMPT_SR)
CLEAN_PROMPT = '/content/creovanta_prompt_16k.wav'
torchaudio.save(CLEAN_PROMPT, audio, PROMPT_SR)
result = cosyvoice.inference_cross_lingual(SYSTEM_PREFIX + TEXT, CLEAN_PROMPT, stream=False, speed=1.0)
speech = result['tts_speech'].cpu()
OUTPUT_PATH = '/content/creovanta_bengali_voice.wav'
torchaudio.save(OUTPUT_PATH, speech, cosyvoice.sample_rate)
print('Audio created:', OUTPUT_PATH, 'Sample rate:', cosyvoice.sample_rate)

In [ ]:
from IPython.display import Audio, display
display(Audio(OUTPUT_PATH))
files.download(OUTPUT_PATH)

## গুরুত্বপূর্ণ
- Colab runtime বন্ধ হলে GPU session বন্ধ হবে; আবার চালাতে মডেল পুনরায় লোড/ডাউনলোড লাগতে পারে।
- এটি এখন পরীক্ষামূলক রান। সফল পরীক্ষা মানেই Creovanta ওয়েব অ্যাপে স্থায়ীভাবে সংযুক্ত হয়েছে—এমন নয়।
- মডেলটি Apache-2.0 লাইসেন্সের Bengali fine-tune; মূল CosyVoice3 প্রকল্প ও fine-tune নির্মাতাকে যথাযথ attribution দিতে হবে।
- অনুমতি ছাড়া কারও কণ্ঠ ক্লোন করবেন না; তৈরি অডিও synthetic speech হিসেবে বিবেচনা করুন।